In [172]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Читаем файл

In [173]:
calls_path = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\Calls (Done).xlsx"
calls = pd.read_excel(calls_path, dtype={"Id": "string", "CONTACTID": "string"})

### Приводим названия колонок к удобному формату

In [174]:
calls.columns = (calls.columns.str.strip().str.lower().str.replace(" ", "_", regex=False).str.replace("(", "", regex=False).str.replace(")", "", regex=False))
calls = calls.rename(columns={"contactid": "contact_id"})
calls.shape

(95874, 11)

In [175]:
calls.head()

,id,call_start_time,call_owner_name,contact_id,call_type,call_duration_in_seconds,call_status,dialled_number,outgoing_call_status,scheduled_in_crm,tag
0,5805028000000805001,30.06.2023 08:43,John Doe,<NA>,Inbound,171.0,Received,NaN,NaN,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,<NA>,Outbound,28.0,Attended Dialled,NaN,Completed,0.0,NaN
2,5805028000000764027,30.06.2023 08:59,John Doe,<NA>,Outbound,24.0,Attended Dialled,NaN,Completed,0.0,NaN
3,5805028000000787003,30.06.2023 09:20,John Doe,5805028000000645014,Outbound,6.0,Attended Dialled,NaN,Completed,0.0,NaN
4,5805028000000768019,30.06.2023 09:30,John Doe,5805028000000645014,Outbound,11.0,Attended Dialled,NaN,Completed,0.0,NaN


In [176]:
calls.dtypes

id                          string[python]
call_start_time                     object
call_owner_name                     object
contact_id                  string[python]
call_type                           object
call_duration_in_seconds           float64
call_status                         object
dialled_number                     float64
outgoing_call_status                object
scheduled_in_crm                   float64
tag                                float64
dtype: object

### Общая статистика по заполненности, типам и уникальности

In [177]:
calls_overview = pd.DataFrame({
    "column": calls.columns,
    "dtype": calls.dtypes.astype(str).values,
    "rows": len(calls),
    "non_null": calls.notna().sum().values,
    "missing": calls.isna().sum().values,
    "missing_pct": (calls.isna().mean().values * 100).round(2),
    "unique_values": calls.nunique(dropna=True).values})

calls_overview

,column,dtype,rows,non_null,missing,missing_pct,unique_values
0,id,string,95874,95874,0,0.00,95874
1,call_start_time,object,95874,95874,0,0.00,68445
2,call_owner_name,object,95874,95874,0,0.00,33
3,contact_id,string,95874,91941,3933,4.10,15214
4,call_type,object,95874,95874,0,0.00,3
5,call_duration_in_seconds,float64,95874,95791,83,0.09,2619
6,call_status,object,95874,95874,0,0.00,11
7,dialled_number,float64,95874,0,95874,100.00,0
8,outgoing_call_status,object,95874,86875,8999,9.39,4
9,scheduled_in_crm,float64,95874,86875,8999,9.39,2


### Предварительно определяем неинформативные столбцы

In [178]:
empty_columns = calls.columns[calls.isna().mean() == 1].tolist()
empty_columns

['dialled_number', 'tag']

In [179]:
# Смотрим долю пропусков по каждому столбцу
missing_columns = pd.DataFrame({
    "column": calls.columns,
    "missing": calls.isna().sum().values,
    "missing_pct": (calls.isna().mean().values * 100).round(2),
    "unique_values": calls.nunique(dropna=True).values})

missing_columns.sort_values("missing_pct", ascending=False)

,column,missing,missing_pct,unique_values
10,tag,95874,100.00,0
7,dialled_number,95874,100.00,0
9,scheduled_in_crm,8999,9.39,2
8,outgoing_call_status,8999,9.39,4
3,contact_id,3933,4.10,15214
5,call_duration_in_seconds,83,0.09,2619
0,id,0,0.00,95874
2,call_owner_name,0,0.00,33
1,call_start_time,0,0.00,68445
6,call_status,0,0.00,11


In [180]:
# Теперь отдельно смотрим кандидатов на удаление:
columns_to_check = ["dialled_number", "tag", "outgoing_call_status", "scheduled_in_crm"]

for col in columns_to_check:
    print(f"\n{col}")
    print(calls[col].value_counts(dropna=False))


dialled_number
dialled_number
NaN    95874
Name: count, dtype: int64

tag
tag
NaN    95874
Name: count, dtype: int64

outgoing_call_status
outgoing_call_status
Completed    86792
NaN           8999
Overdue         60
Cancelled       20
Scheduled        3
Name: count, dtype: int64

scheduled_in_crm
scheduled_in_crm
0.0    86733
NaN     8999
1.0      142
Name: count, dtype: int64


## Удаление неинформативных столбцов
Из таблицы Calls удалены столбцы `dialled_number`, `tag`, `outgoing_call_status`, `scheduled_in_crm`.
* `dialled_number` и `tag` полностью пустые.  
* `outgoing_call_status` является техническим статусом исходящего звонка и для дальнейшего анализа дублирует смысл `call_status`.  
* `scheduled_in_crm` показывает, был ли звонок запланирован в CRM, но в остальных таблицах нет связанных признаков для анализа запланированности. Для целей проекта важнее факт звонка, его тип, статус, длительность, менеджер и связь с контактом.

In [112]:
columns_to_drop = ["dialled_number", "tag", "outgoing_call_status", "scheduled_in_crm"]
calls = calls.drop(columns=columns_to_drop)
calls.head()

,id,call_start_time,call_owner_name,contact_id,call_type,call_duration_in_seconds,call_status
0,5805028000000805001,30.06.2023 08:43,John Doe,<NA>,Inbound,171.0,Received
1,5805028000000768006,30.06.2023 08:46,John Doe,<NA>,Outbound,28.0,Attended Dialled
2,5805028000000764027,30.06.2023 08:59,John Doe,<NA>,Outbound,24.0,Attended Dialled
3,5805028000000787003,30.06.2023 09:20,John Doe,5805028000000645014,Outbound,6.0,Attended Dialled
4,5805028000000768019,30.06.2023 09:30,John Doe,5805028000000645014,Outbound,11.0,Attended Dialled


## Работаем с типами данных

In [181]:
# Преобразуем время начала звонка в datetime
calls["call_start_time"] = pd.to_datetime(calls["call_start_time"], dayfirst=True, errors="coerce")
calls["call_start_time"].dtype

dtype('<M8[ns]')

In [182]:
# Проверяем, все ли даты корректно преобразовались
print("Пустых дат после преобразования:", calls["call_start_time"].isna().sum())
print("Минимальная дата:", calls["call_start_time"].min())
print("Максимальная дата:", calls["call_start_time"].max())

Пустых дат после преобразования: 0
Минимальная дата: 2023-06-30 08:43:00
Максимальная дата: 2024-06-21 15:31:00


In [183]:
# Проверяем длительность звонка
calls["call_duration_in_seconds"].describe()

count    95791.000000
mean       164.977263
std        401.410826
min          0.000000
25%          4.000000
50%          8.000000
75%         98.000000
max       7625.000000
Name: call_duration_in_seconds, dtype: float64

In [184]:
calls[calls["call_duration_in_seconds"].isna()]

,id,call_start_time,call_owner_name,contact_id,call_type,call_duration_in_seconds,call_status,dialled_number,outgoing_call_status,scheduled_in_crm,tag
21711,5805028000012419809,2023-11-15 15:00:00,Kevin Parker,<NA>,Outbound,NaN,Cancelled,NaN,Cancelled,1.0,NaN
36095,5805028000012419971,2024-01-15 15:00:00,Kevin Parker,<NA>,Outbound,NaN,Overdue,NaN,Overdue,1.0,NaN
36096,5805028000012419512,2024-01-15 15:00:00,Kevin Parker,<NA>,Outbound,NaN,Overdue,NaN,Overdue,1.0,NaN
37600,5805028000026957356,2024-01-19 10:00:00,Victor Barnes,<NA>,Outbound,NaN,Overdue,NaN,Overdue,1.0,NaN
37607,5805028000026466529,2024-01-19 10:30:00,Victor Barnes,5805028000021065227,Outbound,NaN,Overdue,NaN,Overdue,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
89073,5805028000053636072,2024-06-05 17:00:00,Victor Barnes,<NA>,Outbound,NaN,Overdue,NaN,Overdue,1.0,NaN
89257,5805028000053651109,2024-06-06 07:00:00,Victor Barnes,<NA>,Outbound,NaN,Cancelled,NaN,Cancelled,1.0,NaN
92959,5805028000055092068,2024-06-14 16:00:00,Victor Barnes,<NA>,Outbound,NaN,Scheduled,NaN,Scheduled,1.0,NaN
93676,5805028000055626456,2024-06-17 10:00:00,Victor Barnes,<NA>,Outbound,NaN,Scheduled,NaN,Scheduled,1.0,NaN


# Работа с пропусками

### Заполняем пропуски в длительности звонка

In [186]:
# Если длительность звонка не заполнена, считаем ее равной 0 секунд. Это безопасно для дальнейшего анализа: такие звонки не будут считаться успешными разговорами.
calls["call_duration_in_seconds"] = (calls["call_duration_in_seconds"].fillna(0).astype("int32"))
calls["call_duration_in_seconds"].isna().sum()

np.int64(0)

### Проверяем пропуски в contact_id

In [187]:
calls["call_has_contact_id"] = calls["contact_id"].notna()

contact_id_missing = calls["contact_id"].isna().sum()
contact_id_missing_pct = round(calls["contact_id"].isna().mean() * 100, 2)

print("Звонков без contact_id:", contact_id_missing)
print("Доля звонков без contact_id:", contact_id_missing_pct, "%")

Звонков без contact_id: 3933
Доля звонков без contact_id: 4.1 %


# Работаем с дубликатами

In [188]:
# Полные дубликаты по всем столбцам
full_duplicates_count = calls.duplicated().sum()
full_duplicates_count

np.int64(0)

In [189]:
# Дубликаты по id
id_duplicates_count = calls["id"].duplicated().sum()
id_duplicates_count

np.int64(0)

In [191]:
# Бизнес-дубли:
# одинаковое время звонка, менеджер, контакт, тип, длительность и статус
duplicate_subset = ["call_start_time", "call_owner_name", "contact_id", "call_type", "call_duration_in_seconds", "call_status"]
business_duplicates = calls[calls.duplicated(subset=duplicate_subset, keep=False)].sort_values(duplicate_subset)
business_duplicates.shape[0]

6382

In [192]:
# Сколько строк будет удалено, если оставить первую запись в каждой группе
calls.duplicated(subset=duplicate_subset, keep="first").sum()

np.int64(3257)

### Удаляем бизнес-дубли

In [193]:
# До удаления фиксируем количество строк
n_before_duplicates = len(calls)
n_before_duplicates

95874

In [194]:
# Удаляем бизнес-дубли.
# Оставляем первую запись в каждой группе одинаковых звонков.
calls = (calls.sort_values("id").drop_duplicates(subset=duplicate_subset, keep="first").reset_index(drop=True))

n_after_duplicates = len(calls)

print("Строк до удаления бизнес-дублей:", n_before_duplicates)
print("Строк после удаления бизнес-дублей:", n_after_duplicates)
print("Удалено строк:", n_before_duplicates - n_after_duplicates)

Строк до удаления бизнес-дублей: 95874
Строк после удаления бизнес-дублей: 92617
Удалено строк: 3257


In [195]:
calls.duplicated(subset=duplicate_subset).sum()

np.int64(0)

## Вывод по дубликатам
* В таблице Calls не обнаружено полных дублей и дублей по `id`, то есть каждая запись звонка имеет уникальный CRM ID.
* Однако были найдены бизнес-дубли: записи с одинаковым временем звонка, менеджером, контактом, типом звонка, длительностью и статусом. Такие строки можно считать повторной фиксацией одного и того же звонка в CRM.
* Для очистки данных оставлена первая запись в каждой группе бизнес-дублей, остальные удалены. Это нужно, чтобы не завышать количество звонков, активность менеджеров и метрики дозвона.

### Проверяем пересекающиеся звонки у одного менеджера

In [196]:
# Рассчитываем время окончания звонка
calls["call_end_time"] = (calls["call_start_time"] + pd.to_timedelta(calls["call_duration_in_seconds"], unit="s"))

# Сортируем звонки по менеджеру и времени начала
calls_sorted = calls.sort_values(["call_owner_name", "call_start_time"]).copy()

# Для каждого менеджера берем время окончания предыдущего звонка
calls_sorted["prev_call_end_time"] = (calls_sorted.groupby("call_owner_name")["call_end_time"].shift(1))

# Если текущий звонок начался раньше, чем закончился предыдущий, значит есть пересечение по времени
overlapping_calls = calls_sorted[calls_sorted["call_start_time"] < calls_sorted["prev_call_end_time"]].copy()

overlapping_calls["overlap_seconds"] = (overlapping_calls["prev_call_end_time"] - overlapping_calls["call_start_time"]).dt.total_seconds()

print("Строк с пересечением времени:", len(overlapping_calls))
print("Доля пересекающихся звонков:", round(len(overlapping_calls) / len(calls) * 100, 2), "%")

overlapping_calls["overlap_seconds"].describe()

Строк с пересечением времени: 8128
Доля пересекающихся звонков: 8.78 %


count    8128.000000
mean      120.663140
std       346.098052
min         1.000000
25%         5.000000
50%         8.000000
75%        28.000000
max      7140.000000
Name: overlap_seconds, dtype: float64

In [197]:
# Распределение пересечений по длительности
overlap_groups = pd.cut(overlapping_calls["overlap_seconds"],bins=[0, 5, 15, 60, 300, 1800, np.inf], labels=[
        "до 5 сек",
        "6-15 сек",
        "16-60 сек",
        "1-5 мин",
        "5-30 мин",
        "более 30 мин"])

overlap_groups.value_counts().sort_index()

overlap_seconds
до 5 сек        2295
6-15 сек        3111
16-60 сек       1095
1-5 мин          736
5-30 мин         814
более 30 мин      77
Name: count, dtype: int64

## Вывод по пересекающимся звонкам
* Пересекающиеся звонки — это ситуации, когда новый звонок у того же менеджера начинается раньше, чем закончился предыдущий.
* Такие строки не удаляем автоматически, потому что пересечение по времени не доказывает, что запись является дублем. Возможные причины: технические особенности CRM, сбои, автодозвон, задержка закрытия карточки звонка, параллельные линии или ошибка логирования длительности.
* Эта проверка нужна для понимания качества данных. Для удаления строк используем только более строгую логику бизнес-дублей: совпадение времени звонка, менеджера, контакта, типа звонка, длительности и статуса.

# Применяем mapping контактов
Если мы просто удалим дубль из Contacts, то в других таблицах может остаться ссылка на удаленный id.
`применяем mapping из очищенных Contacts, чтобы звонки, которые ссылаются на удаленные contact_id-дубли, перепривязались к master-contact.`


In [198]:
contacts_mapping_path = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\дополнительные файлы\contacts_duplicates_mapping.csv"
contacts_mapping = pd.read_csv(contacts_mapping_path, dtype={"old_contact_id": "string", "master_contact_id": "string"})
contacts_mapping.head()

,old_contact_id,master_contact_id,contact_owner_name,created_time,modified_time
0,5805028000044202223,5805028000044178663,Amy Green,2024-04-16 18:13:00,2024-04-16 20:13:00
1,5805028000053717839,5805028000053716641,Ben Hall,2024-06-06 17:48:00,2024-06-06 19:48:00
2,5805028000001953081,5805028000001949074,Bob Brown,2023-07-17 18:40:00,2023-07-17 18:40:00
3,5805028000002344049,5805028000002340967,Bob Brown,2023-07-18 16:53:00,2023-07-18 16:53:00
4,5805028000002782035,5805028000002740077,Bob Brown,2023-07-21 12:26:00,2023-07-21 12:26:00


In [199]:
# Создаем словарь для замены old_contact_id на master_contact_id
contact_id_mapping = dict(zip(contacts_mapping["old_contact_id"], contacts_mapping["master_contact_id"]))
len(contact_id_mapping)

38

In [200]:
# Проверяем, сколько звонков будет перепривязано
affected_calls = calls["contact_id"].isin(contact_id_mapping.keys()).sum()
affected_calls

np.int64(102)

In [201]:
# Применяем mapping
calls["contact_id"] = calls["contact_id"].replace(contact_id_mapping)
print("Перепривязано звонков:", affected_calls)

Перепривязано звонков: 102


In [202]:
# Проверяем, что старых contact_id из mapping больше нет
calls["contact_id"].isin(contact_id_mapping.keys()).sum()

np.int64(0)

## Вывод по mapping контактов
После очистки таблицы Contacts часть contact_id была признана бизнес-дублями и заменена на master_contact_id. Чтобы не потерять связь звонков с контактами, к таблице Calls был применен mapping `old_contact_id -> master_contact_id`. Это позволяет сохранить звонки, которые были связаны со старыми дублями контактов, и корректно учитывать их при дальнейшем объединении Calls с Contacts и Deals.

### Создаем дополнительные признаки для анализа звонков

In [204]:
# Дата звонка без времени
calls["call_date"] = calls["call_start_time"].dt.date

In [205]:
# Час звонка
calls["call_hour"] = calls["call_start_time"].dt.hour

In [206]:
# Длительность звонка в минутах
calls["call_duration_min"] = (calls["call_duration_in_seconds"] / 60).round(2)

In [207]:
# Флаг успешного звонка.
# Успешным считаем звонок, где был реальный контакт с клиентом:
# - исходящий дозвон: Attended Dialled
# - входящий принятый: Received
# Дополнительно ставим порог длительности > 5 секунд, чтобы не считать технические короткие соединения полноценным разговором.

calls["is_successful_call"] = (calls["call_status"].isin(["Attended Dialled", "Received"]) & (calls["call_duration_in_seconds"] > 5))

In [208]:
# Флаг исходящего звонка
calls["is_outbound_call"] = calls["call_type"].eq("Outbound")

In [209]:
# Флаг входящего принятого звонка
calls["is_inbound_call"] = calls["call_type"].eq("Inbound")

In [210]:
# Флаг пропущенного звонка
calls["is_missed_call"] = calls["call_type"].eq("Missed")

In [211]:
# Проверяем новые признаки
calls[["call_start_time", "call_date", "call_hour", "call_duration_in_seconds", "call_duration_min", "call_type", "call_status", "is_successful_call", "is_outbound_call", "is_inbound_call", "is_missed_call"]].head()

,call_start_time,call_date,call_hour,call_duration_in_seconds,call_duration_min,call_type,call_status,is_successful_call,is_outbound_call,is_inbound_call,is_missed_call
0,2023-06-30 08:59:00,2023-06-30,8,24,0.40,Outbound,Attended Dialled,True,True,False,False
1,2023-06-30 08:46:00,2023-06-30,8,28,0.47,Outbound,Attended Dialled,True,True,False,False
2,2023-06-30 09:30:00,2023-06-30,9,11,0.18,Outbound,Attended Dialled,True,True,False,False
3,2023-06-30 14:24:00,2023-06-30,14,4,0.07,Outbound,Attended Dialled,False,True,False,False
4,2023-06-30 09:20:00,2023-06-30,9,6,0.10,Outbound,Attended Dialled,True,True,False,False


In [212]:
# Краткая проверка успешных звонков
successful_calls_count = calls["is_successful_call"].sum()
successful_calls_pct = round(calls["is_successful_call"].mean() * 100, 2)

print("Успешных звонков:", successful_calls_count)
print("Доля успешных звонков:", successful_calls_pct, "%")

Успешных звонков: 57553
Доля успешных звонков: 62.14 %


### Вывод по дополнительным признакам
Для дальнейшего анализа звонков были добавлены производные признаки: дата звонка, час звонка, длительность в минутах, а также флаги исходящих, входящих, пропущенных и успешных звонков.
Успешным звонком считаем звонок со статусом `Attended Dialled` или `Received` и длительностью больше 5 секунд. Это позволяет отделить реальные разговоры от технических коротких соединений и неуспешных попыток дозвона.

### Отображение работы менеджеров

In [221]:
manager_calls_summary = (calls.groupby("call_owner_name", dropna=False).agg(
        total_calls=("id", "count"),
        unique_contacts=("contact_id", "nunique"),
        missing_contact_id=("contact_id", lambda x: x.isna().sum()),
        avg_duration_sec=("call_duration_in_seconds", "mean"),
        median_duration_sec=("call_duration_in_seconds", "median")).reset_index())

manager_calls_summary.head()

,call_owner_name,total_calls,unique_contacts,missing_contact_id,avg_duration_sec,median_duration_sec
0,Alice Johnson,1244,332,24,113.478296,6.0
1,Amy Green,5575,2559,142,103.671749,15.0
2,Ben Hall,2850,684,74,256.599298,9.0
3,Bob Brown,98,61,5,268.336735,26.5
4,Cara Iverson,3019,1204,5,168.830739,8.0


In [222]:
call_type_by_manager = (calls.pivot_table(index="call_owner_name", columns="call_type", values="id", aggfunc="count", fill_value=0).reset_index())
call_type_by_manager.head()

call_type,call_owner_name,Inbound,Missed,Outbound
0,Alice Johnson,14,3,1227
1,Amy Green,142,281,5152
2,Ben Hall,0,0,2850
3,Bob Brown,0,0,98
4,Cara Iverson,0,0,3019


In [223]:
# Добавим типы звонков отдельными столбцами
manager_calls_summary = manager_calls_summary.merge(call_type_by_manager, on="call_owner_name", how="left")
manager_calls_summary.head()

,call_owner_name,total_calls,unique_contacts,missing_contact_id,avg_duration_sec,median_duration_sec,Inbound,Missed,Outbound
0,Alice Johnson,1244,332,24,113.478296,6.0,14,3,1227
1,Amy Green,5575,2559,142,103.671749,15.0,142,281,5152
2,Ben Hall,2850,684,74,256.599298,9.0,0,0,2850
3,Bob Brown,98,61,5,268.336735,26.5,0,0,98
4,Cara Iverson,3019,1204,5,168.830739,8.0,0,0,3019


In [224]:
# добавляем анализ эффективности
manager_success = (calls.groupby("call_owner_name", dropna=False).agg(successful_calls=("is_successful_call", "sum"), success_rate=("is_successful_call", "mean")).reset_index())
manager_calls_summary = manager_calls_summary.merge(manager_success, on="call_owner_name", how="left")
manager_calls_summary["success_rate"] = (manager_calls_summary["success_rate"] * 100).round(2)
manager_calls_summary.head(30)

,call_owner_name,total_calls,unique_contacts,missing_contact_id,avg_duration_sec,median_duration_sec,Inbound,Missed,Outbound,successful_calls,success_rate
0,Alice Johnson,1244,332,24,113.478296,6.0,14,3,1227,645,51.85
1,Amy Green,5575,2559,142,103.671749,15.0,142,281,5152,3892,69.81
2,Ben Hall,2850,684,74,256.599298,9.0,0,0,2850,2119,74.35
3,Bob Brown,98,61,5,268.336735,26.5,0,0,98,82,83.67
4,Cara Iverson,3019,1204,5,168.830739,8.0,0,0,3019,1789,59.26
5,Charlie Davis,6943,1739,363,225.158145,8.0,287,607,6049,4396,63.32
6,Derek James,941,256,87,195.721573,88.0,94,0,847,764,81.19
7,Diana Evans,6713,1190,182,187.256368,7.0,280,319,6114,3926,58.48
8,Ethan Harris,277,261,6,50.494585,8.0,0,4,273,217,78.34
9,Eva Kent,492,212,12,403.308943,6.0,8,5,479,266,54.07


### Итоговая проверка после очистки Calls

In [226]:
calls_summary = pd.DataFrame({
    "metric": ["Строк после очистки", "Уникальных звонков id", "Дубликатов по id", "Полных дублей", "Бизнес-дублей", "Звонков без contact_id",
        "Доля звонков без contact_id, %", "Успешных звонков", "Доля успешных звонков, %", "Уникальных менеджеров", "Минимальная дата звонка",
        "Максимальная дата звонка", "Пропусков всего"], 
    "value": [len(calls), calls["id"].nunique(), calls["id"].duplicated().sum(), calls.duplicated().sum(), calls.duplicated(subset=duplicate_subset).sum(),
        calls["contact_id"].isna().sum(), round(calls["contact_id"].isna().mean() * 100, 2), calls["is_successful_call"].sum(), round(calls["is_successful_call"].mean() * 100, 2),
        calls["call_owner_name"].nunique(), calls["call_start_time"].min(), calls["call_start_time"].max(), calls.isna().sum().sum()]})

calls_summary

,metric,value
0,Строк после очистки,92617
1,Уникальных звонков id,92617
2,Дубликатов по id,0
3,Полных дублей,0
4,Бизнес-дублей,0
5,Звонков без contact_id,3802
6,"Доля звонков без contact_id, %",4.11
7,Успешных звонков,57553
8,"Доля успешных звонков, %",62.14
9,Уникальных менеджеров,33


#### В основной очищенный файл оставлены только исходные аналитически значимые поля и один дополнительный признак `is_successful_call`. 
Технические поля, использованные для проверки пересечений звонков или удобства промежуточного анализа, не включались в финальный файл. При необходимости `call_date`, `call_hour`, `call_duration_min` и другие признаки можно создать на этапе EDA или в Power BI.

In [227]:
final_calls_columns = ["id", "call_start_time", "call_owner_name", "contact_id", "call_type", "call_duration_in_seconds", "call_status", "is_successful_call"]
calls_clean = calls[final_calls_columns].copy()
calls_clean.shape

(92617, 8)

In [228]:
calls_clean.head()

,id,call_start_time,call_owner_name,contact_id,call_type,call_duration_in_seconds,call_status,is_successful_call
0,5805028000000764027,2023-06-30 08:59:00,John Doe,<NA>,Outbound,24,Attended Dialled,True
1,5805028000000768006,2023-06-30 08:46:00,John Doe,<NA>,Outbound,28,Attended Dialled,True
2,5805028000000768019,2023-06-30 09:30:00,John Doe,5805028000000645014,Outbound,11,Attended Dialled,True
3,5805028000000773022,2023-06-30 14:24:00,John Doe,5805028000000645014,Outbound,4,Attended Dialled,False
4,5805028000000787003,2023-06-30 09:20:00,John Doe,5805028000000645014,Outbound,6,Attended Dialled,True


# Сохраняем результат

In [229]:
# Сохраняем очищенный Calls и сводку по менеджерам
output_dir = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\дополнительные файлы"
os.makedirs(output_dir, exist_ok=True)
calls_clean.to_csv(os.path.join(output_dir, "calls_clean.csv"), index=False, encoding="utf-8-sig")
calls_clean.to_excel(os.path.join(output_dir, "calls_clean.xlsx"), index=False)
manager_calls_summary.to_csv(os.path.join(output_dir, "manager_calls_summary.csv"), index=False, encoding="utf-8-sig")
calls_summary.to_csv(os.path.join(output_dir, "calls_cleaning_summary.csv"), index=False, encoding="utf-8-sig")

# Вывод по очистке Calls

* В таблице Calls было 95 874 строки и 11 столбцов.  
CRM ID (`id` и `contact_id`) были сохранены в строковом формате, чтобы избежать потери точности при дальнейших объединениях таблиц.

* В ходе очистки были удалены неинформативные столбцы `dialled_number`, `tag`, `outgoing_call_status`, `scheduled_in_crm`.  
`dialled_number` и `tag` полностью пустые, а `outgoing_call_status` и `scheduled_in_crm` являются техническими признаками, которые не используются в дальнейшей аналитике воронки, юнит-экономики и эффективности продаж.

* Столбец `call_start_time` был преобразован в формат datetime.  
Пропуски в `call_duration_in_seconds` были заменены на 0, так как такие звонки не могут считаться успешными разговорами.

* Полных дублей и дублей по `id` не обнаружено.  
Были найдены и удалены бизнес-дубли: звонки с одинаковым временем, менеджером, контактом, типом, длительностью и статусом. Это сделано, чтобы не завышать количество звонков и активность менеджеров.

* Пропуски в `contact_id` не заполнялись вручную, потому что надежной связи для восстановления контакта нет. Вместо этого был создан флаг `call_has_contact_id`.

* Также был применен mapping из очищенной таблицы Contacts, чтобы звонки, связанные со старыми дублями контактов, были перепривязаны к master-contact.

* Дополнительно были созданы признаки `call_date`, `call_hour`, `call_duration_min`, `is_successful_call`, `is_outbound_call`, `is_inbound_call`, `is_missed_call`. Успешным звонком считается звонок со статусом `Attended Dialled` или `Received` и длительностью больше 5 секунд. Но в финальный файл данные столбцы не включались так как они нужны только для анализа. В таблицу вклчил только признак - `is_successful_call`.

* Для дальнейшего анализа была создана сводка по менеджерам `manager_calls_summary`, где рассчитаны количество звонков, количество уникальных контактов, пропуски contact_id, средняя и медианная длительность звонка, типы звонков и доля успешных звонков.